# Full model evaluation — predicted vs actual final sentence

Evaluates the deployed prediction pipeline (`predictorBackend/src/predictor.ts`
on top of `predictorBackend/src/guidelineModel.ts`) against the **verified**
final sentences in the shared cache. The full model applies, in order:

1. **Starting point** — pure-guideline interpolation per drug family, combined
   across drugs with the HK notional-quantity (total-quantity weighted) method.
2. **Role adjustment** (and role cross-border), from the reviewed role workbook.
3. **Aggravating factors** — the five supported factors, as percentages.
4. **Mitigating factors** — including assistance, applied **before** the plea.
5. **Guilty plea** discount — guideline percentage by plea stage, applied **last**.

The predicted final sentence is compared with the trial's verified
`final_sentence.total_months`. Only trials with a non-inferred final sentence are
included; trials containing a drug the model does not support are excluded from
coverage.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

notebook_dir = Path.cwd().resolve()
shared_dir = notebook_dir if (notebook_dir / ".cache").exists() else notebook_dir.parent
if str(shared_dir) not in sys.path:
	sys.path.insert(0, str(shared_dir))

from linear_interpolation_model import (
	clean_quantity,
	flatten_documents,
	load_documents,
	total_months,
	trial_catalogue_key,
)

ROLE_WORKBOOK = "Role Sentence Adjustments_updated 2026.07.30.xlsx"

documents, cache_metadata = load_documents(shared_dir, refresh_cache=False)
trial_rows, _effect_rows = flatten_documents(documents)

trial_rows = trial_rows.loc[
	~trial_rows["source_document_excluded"]
	& trial_rows["final_sentence_months"].notna()
	& ~trial_rows["final_sentence_inferred"]
].copy()

workbook = pd.read_excel(notebook_dir / ROLE_WORKBOOK)
workbook = workbook.copy()
workbook["role_catalogue_key"] = workbook.apply(
	lambda row: trial_catalogue_key(
		row["neutral_citation"],
		row["trial_index"],
		row["Charge_no"],
		row["Defendant_id"],
	),
	axis=1,
)
workbook["workbook_excluded"] = (
	pd.to_numeric(workbook["Exclusion"], errors="coerce").fillna(0).eq(1)
)
workbook_role_map = {
	"Actual trafficker": "Actual trafficker",
	"Manager/organiser": "Manager / Organiser",
	"Operator/financial controller": "Operator / Financial Controller",
}
workbook["model_role"] = (
	workbook["Defendant's Role"]
	.astype(str)
	.str.strip()
	.map(workbook_role_map)
)
workbook["model_circumstances"] = (
	workbook["Additional Circumstances"]
	.astype(str)
	.eq("Cross-border trafficking")
	.map(lambda value: ["Cross-border trafficking"] if value else [])
)
role_rows = (
	workbook.loc[
		~workbook["workbook_excluded"] & workbook["model_role"].notna(),
		["role_catalogue_key", "model_role", "model_circumstances"],
	]
	.drop_duplicates("role_catalogue_key")
)

trial_rows = trial_rows.merge(
	role_rows,
	on="role_catalogue_key",
	how="left",
	validate="many_to_one",
)

print(f"eligible trials: {len(trial_rows)}")
print(f"trials with a reviewed role: {trial_rows['model_role'].notna().sum()}")


eligible trials: 2599
trials with a reviewed role: 110


In [2]:
# Port of predictorBackend/src/guidelineModel.ts — pure-guideline starting point.

DRUG_FAMILY_MAP = {
	'Cocaine': 'Cocaine',
	'Ketamine': 'Ketamine',
	'Fluorodeschloroketamine': 'Ketamine',
	'Methamphetamine': 'Methamphetamine',
	'Heroin': 'Heroin',
	'Cannabis/THC': 'Cannabis',
	'Ecstasy': 'Ecstasy',
	'Nimetazepam': 'Nimetazepam',
}

def drug_family_for(drug_type, variant=None):
	if drug_type == 'Midazolam':
		return 'Midazolam-powder' if variant == 'powder' else None
	return DRUG_FAMILY_MAP.get(drug_type)

GUIDELINE_BUCKETS = {'Cocaine': [{'low_q': 0, 'high_q': 10, 'low_s': 24, 'high_s': 60, 'kind': 'bounded'}, {'low_q': 10, 'high_q': 50, 'low_s': 60, 'high_s': 96, 'kind': 'bounded'}, {'low_q': 50, 'high_q': 200, 'low_s': 96, 'high_s': 144, 'kind': 'bounded'}, {'low_q': 200, 'high_q': 500, 'low_s': 144, 'high_s': 192, 'kind': 'bounded'}, {'low_q': 500, 'high_q': 1500, 'low_s': 192, 'high_s': 240, 'kind': 'bounded'}, {'low_q': 1500, 'high_q': 5000, 'low_s': 240, 'high_s': 288, 'kind': 'bounded'}, {'low_q': 5000, 'high_q': 15000, 'low_s': 288, 'high_s': 324, 'kind': 'bounded'}, {'low_q': 15000, 'high_q': 30000, 'low_s': 324, 'high_s': 360, 'kind': 'bounded'}, {'low_q': 30000, 'high_q': None, 'low_s': 0, 'high_s': 420, 'kind': 'discretionTop'}], 'Ketamine': [{'low_q': 0, 'high_q': 1, 'low_s': 0, 'high_s': 24, 'kind': 'discretionStart'}, {'low_q': 1, 'high_q': 10, 'low_s': 24, 'high_s': 48, 'kind': 'bounded'}, {'low_q': 10, 'high_q': 50, 'low_s': 48, 'high_s': 72, 'kind': 'bounded'}, {'low_q': 50, 'high_q': 300, 'low_s': 72, 'high_s': 108, 'kind': 'bounded'}, {'low_q': 300, 'high_q': 600, 'low_s': 108, 'high_s': 144, 'kind': 'bounded'}, {'low_q': 600, 'high_q': 1000, 'low_s': 144, 'high_s': 168, 'kind': 'bounded'}, {'low_q': 1000, 'high_q': 2000, 'low_s': 168, 'high_s': 216, 'kind': 'bounded'}, {'low_q': 2000, 'high_q': 3000, 'low_s': 216, 'high_s': 240, 'kind': 'bounded'}, {'low_q': 3000, 'high_q': None, 'low_s': 240, 'high_s': None, 'kind': 'openUp'}], 'Methamphetamine': [{'low_q': 0, 'high_q': 10, 'low_s': 36, 'high_s': 84, 'kind': 'bounded'}, {'low_q': 10, 'high_q': 70, 'low_s': 84, 'high_s': 132, 'kind': 'bounded'}, {'low_q': 70, 'high_q': 300, 'low_s': 132, 'high_s': 180, 'kind': 'bounded'}, {'low_q': 300, 'high_q': 600, 'low_s': 180, 'high_s': 216, 'kind': 'bounded'}, {'low_q': 600, 'high_q': 1500, 'low_s': 216, 'high_s': 240, 'kind': 'bounded'}, {'low_q': 1500, 'high_q': 5000, 'low_s': 240, 'high_s': 288, 'kind': 'bounded'}, {'low_q': 5000, 'high_q': 15000, 'low_s': 288, 'high_s': 324, 'kind': 'bounded'}, {'low_q': 15000, 'high_q': 30000, 'low_s': 324, 'high_s': 360, 'kind': 'bounded'}, {'low_q': 30000, 'high_q': None, 'low_s': 0, 'high_s': 420, 'kind': 'discretionTop'}], 'Heroin': [{'low_q': 0, 'high_q': 10, 'low_s': 24, 'high_s': 60, 'kind': 'bounded'}, {'low_q': 10, 'high_q': 50, 'low_s': 60, 'high_s': 96, 'kind': 'bounded'}, {'low_q': 50, 'high_q': 200, 'low_s': 96, 'high_s': 144, 'kind': 'bounded'}, {'low_q': 200, 'high_q': 500, 'low_s': 144, 'high_s': 192, 'kind': 'bounded'}, {'low_q': 500, 'high_q': 1500, 'low_s': 192, 'high_s': 240, 'kind': 'bounded'}, {'low_q': 1500, 'high_q': 5000, 'low_s': 240, 'high_s': 288, 'kind': 'bounded'}, {'low_q': 5000, 'high_q': 15000, 'low_s': 288, 'high_s': 324, 'kind': 'bounded'}, {'low_q': 15000, 'high_q': 30000, 'low_s': 324, 'high_s': 360, 'kind': 'bounded'}, {'low_q': 30000, 'high_q': None, 'low_s': 0, 'high_s': 420, 'kind': 'discretionTop'}], 'Cannabis': [{'low_q': 0, 'high_q': 2000, 'low_s': 0, 'high_s': 16, 'kind': 'bounded'}, {'low_q': 2000, 'high_q': 3000, 'low_s': 16, 'high_s': 24, 'kind': 'bounded'}, {'low_q': 3000, 'high_q': 6000, 'low_s': 24, 'high_s': 36, 'kind': 'bounded'}, {'low_q': 6000, 'high_q': 9000, 'low_s': 36, 'high_s': 48, 'kind': 'bounded'}, {'low_q': 9000, 'high_q': 15000, 'low_s': 48, 'high_s': 66, 'kind': 'bounded'}, {'low_q': 15000, 'high_q': 45000, 'low_s': 66, 'high_s': 96, 'kind': 'bounded'}, {'low_q': 45000, 'high_q': 90000, 'low_s': 96, 'high_s': 120, 'kind': 'bounded'}, {'low_q': 90000, 'high_q': None, 'low_s': 120, 'high_s': None, 'kind': 'openUp'}], 'Ecstasy': [{'low_q': 0, 'high_q': 1, 'low_s': 0, 'high_s': 24, 'kind': 'discretionStart'}, {'low_q': 1, 'high_q': 10, 'low_s': 24, 'high_s': 48, 'kind': 'bounded'}, {'low_q': 10, 'high_q': 50, 'low_s': 48, 'high_s': 72, 'kind': 'bounded'}, {'low_q': 50, 'high_q': 300, 'low_s': 72, 'high_s': 108, 'kind': 'bounded'}, {'low_q': 300, 'high_q': 600, 'low_s': 108, 'high_s': 144, 'kind': 'bounded'}, {'low_q': 600, 'high_q': 1000, 'low_s': 144, 'high_s': 168, 'kind': 'bounded'}, {'low_q': 1000, 'high_q': 2000, 'low_s': 168, 'high_s': 216, 'kind': 'bounded'}, {'low_q': 2000, 'high_q': 3000, 'low_s': 216, 'high_s': 240, 'kind': 'bounded'}, {'low_q': 3000, 'high_q': None, 'low_s': 240, 'high_s': None, 'kind': 'openUp'}], 'Nimetazepam': [{'low_q': 0, 'high_q': 1, 'low_s': 0, 'high_s': 24, 'kind': 'discretionStart'}, {'low_q': 1, 'high_q': 10, 'low_s': 24, 'high_s': 48, 'kind': 'bounded'}, {'low_q': 10, 'high_q': 50, 'low_s': 48, 'high_s': 72, 'kind': 'bounded'}, {'low_q': 50, 'high_q': 300, 'low_s': 72, 'high_s': 108, 'kind': 'bounded'}, {'low_q': 300, 'high_q': 600, 'low_s': 108, 'high_s': 144, 'kind': 'bounded'}, {'low_q': 600, 'high_q': 1000, 'low_s': 144, 'high_s': 168, 'kind': 'bounded'}, {'low_q': 1000, 'high_q': 2000, 'low_s': 168, 'high_s': 216, 'kind': 'bounded'}, {'low_q': 2000, 'high_q': 3000, 'low_s': 216, 'high_s': 240, 'kind': 'bounded'}, {'low_q': 3000, 'high_q': None, 'low_s': 240, 'high_s': None, 'kind': 'openUp'}], 'Midazolam-powder': [{'low_q': 0, 'high_q': 500, 'low_s': 0, 'high_s': 6, 'kind': 'discretionStart'}, {'low_q': 500, 'high_q': 1000, 'low_s': 6, 'high_s': 12, 'kind': 'bounded'}, {'low_q': 1000, 'high_q': 2000, 'low_s': 12, 'high_s': 24, 'kind': 'bounded'}, {'low_q': 2000, 'high_q': 3000, 'low_s': 24, 'high_s': 36, 'kind': 'bounded'}, {'low_q': 3000, 'high_q': 6000, 'low_s': 36, 'high_s': 54, 'kind': 'bounded'}, {'low_q': 6000, 'high_q': 9000, 'low_s': 54, 'high_s': 72, 'kind': 'bounded'}, {'low_q': 9000, 'high_q': None, 'low_s': 72, 'high_s': None, 'kind': 'openUp'}]}

def interpolate(bucket, quantity, previous_high_s):
	if bucket["high_q"] is not None and bucket["high_s"] is not None:
		u = (quantity - bucket["low_q"]) / (bucket["high_q"] - bucket["low_q"])
		return bucket["low_s"] + u * (bucket["high_s"] - bucket["low_s"])
	if bucket["kind"] == "discretionTop" and previous_high_s is not None:
		return previous_high_s
	return bucket["low_s"]

def predict_starting_point_months(drug_type, quantity, variant=None):
	family = drug_family_for(drug_type, variant)
	if family is None:
		return None
	previous_high_s = None
	for bucket in GUIDELINE_BUCKETS[family]:
		if bucket["high_q"] is not None:
			if bucket["low_q"] <= quantity < bucket["high_q"]:
				return interpolate(bucket, quantity, previous_high_s)
		else:
			if quantity >= bucket["low_q"]:
				return interpolate(bucket, quantity, previous_high_s)
		if bucket["high_s"] is not None:
			previous_high_s = bucket["high_s"]
	return None

def predict_notional_weighted_months(drugs):
	"""HK notional-quantity method (predictorBackend/src/guidelineModel.ts)."""
	total = sum(drug["quantity"] for drug in drugs)
	if total <= 0:
		return 0.0
	starting_point = 0.0
	for drug in drugs:
		sentence_at_total = predict_starting_point_months(
			drug["type"], total, drug.get("variant")
		)
		if sentence_at_total is None:
			return None
		starting_point += sentence_at_total * (drug["quantity"] / total)
	return starting_point

# Sanity checks against the predictorBackend test expectations.
def round2(value):
	return round(value + 1e-9, 2)
assert abs(predict_starting_point_months('Cocaine', 10) - 60) < 1e-9
assert abs(round2(predict_notional_weighted_months([
	{'type': 'Fluorodeschloroketamine', 'quantity': 2},
	{'type': 'Heroin', 'quantity': 1},
])) - 31.16) < 1e-6
assert abs(predict_starting_point_months('Midazolam', 2, 'powder') - 0.024) < 1e-9
print('guideline model port matches the predictorBackend test values')


guideline model port matches the predictorBackend test values


In [3]:
# Port of predictorBackend/src/predictor.ts — full sentence pipeline.

ROLE_ADJUSTMENTS = {
	'Courier / Storekeeper': 0,
	'Actual trafficker': 0.05,
	'Manager / Organiser': 0.06,
	'Operator / Financial Controller': 0.08,
}

AGGRAVATING_ADJUSTMENTS = {
	'Multiple Drugs': 0.0385,
	'Persistent offender': 0.04,
	'On bail': 0.0441,
	'Refugee/Asylum': 0.08,
	'Use of minors': 0.0525,
}

COURIER_CROSS_BORDER_ADJUSTMENT = 0.0588
ROLE_CROSS_BORDER_ADJUSTMENTS = {
	'Actual trafficker': 0.29,
	'Manager / Organiser': 0.08,
	'Operator / Financial Controller': 0.1,
}

MITIGATING_ADJUSTMENTS = {
	'Self-consumption': 0.0451,
	'Assistance - limited': 0.0182,
	'Assistance - useful': 0.05,
	'Assistance - testify': 0.3108,
	'Assistance - risk': 0.0448,
	'Young offender': 0.0409,
	'Medical conditions': 0.03,
	'Family illness': 0.03,
	'Rehabilitation programme': 0.0114,
}

GUILTY_PLEA_ADJUSTMENTS = {
	'Plead guilty (earliest opportunity)': 0.333,
	'Plead guilty (before trial dates are set)': 0.25,
	'Plead guilty (before trial starts)': 0.225,
	'Plead guilty (first day of trial)': 0.2,
	'Plead guilty (during the trial)': 0.15,
}

def round2(value):
	return round(value + 1e-9, 2)

def predict_sentence(drugs, defendant_role, additional_circumstances,
					 guilty_plea, aggravating_factors, mitigating_factors):
	"""Returns stage months and the rounded final sentence, or None if unsupported."""
	starting_point = predict_notional_weighted_months(drugs)
	if starting_point is None:
		return None
	current = starting_point
	role_months = 0.0
	if isinstance(defendant_role, str):
		is_courier = defendant_role == 'Courier / Storekeeper'
		has_cross_border = 'Cross-border trafficking' in additional_circumstances
		if not is_courier and has_cross_border:
			role_months = current * ROLE_CROSS_BORDER_ADJUSTMENTS[defendant_role]
		else:
			role_months = current * ROLE_ADJUSTMENTS[defendant_role]
			if has_cross_border:
				current += current * COURIER_CROSS_BORDER_ADJUSTMENT
		current += role_months
	after_role = current
	aggravating_months = sum(
		current * AGGRAVATING_ADJUSTMENTS[factor]
		for factor in aggravating_factors
	)
	current += aggravating_months
	notional = current
	mitigating_months = sum(
		current * MITIGATING_ADJUSTMENTS[factor]
		for factor in mitigating_factors
		if factor in MITIGATING_ADJUSTMENTS
	)
	current -= mitigating_months
	pre_plea = current
	plea_months = (
		current * GUILTY_PLEA_ADJUSTMENTS[guilty_plea]
		if guilty_plea is not None and guilty_plea in GUILTY_PLEA_ADJUSTMENTS
		else 0.0
	)
	current -= plea_months
	return {
		'starting_point_months': starting_point,
		'role_months': role_months,
		'sentence_after_role_months': after_role,
		'aggravating_months': aggravating_months,
		'notional_months': notional,
		'mitigating_months': mitigating_months,
		'pre_plea_months': pre_plea,
		'plea_months': plea_months,
		'final_sentence_months': round2(max(0.0, current)),
	}

# Sanity check against the deterministic predictorBackend test.
result = predict_sentence(
	drugs=[{'type': 'Cocaine', 'quantity': 10}],
	defendant_role='Actual trafficker',
	additional_circumstances=[],
	guilty_plea='Plead guilty (earliest opportunity)',
	aggravating_factors=['Multiple Drugs'],
	mitigating_factors=['Assistance - useful'],
)
assert abs(result['starting_point_months'] - 60) < 1e-9
assert abs(result['final_sentence_months'] - 41.46) < 1e-9
print('predictor port matches the predictorBackend deterministic test (41.46 months)')


predictor port matches the predictorBackend deterministic test (41.46 months)


In [4]:
# Verified feature -> model input mapping.

DRUG_VERIFIED_TO_MODEL = {
	'Cocaine': 'Cocaine',
	'Heroin': 'Heroin',
	'Methamphetamine': 'Methamphetamine',
	'Ketamine': 'Ketamine',
	'Fluorodeschloroketamine': 'Fluorodeschloroketamine',
	'Nimetazepam': 'Nimetazepam',
	'Ecstasy': 'Ecstasy',
	'Cannabis': 'Cannabis/THC',
	'THC/CBD': 'Cannabis/THC',
}

AGGRAVATING_MODEL_MAP = {
	'Multiple drugs': 'Multiple Drugs',
	'Persistent offender': 'Persistent offender',
	'On bail': 'On bail',
	'Refugee claimant': 'Refugee/Asylum',
	'Use of minors': 'Use of minors',
}

MITIGATING_MODEL_FACTORS = {
	'Self-consumption',
	'Assistance - limited',
	'Assistance - useful',
	'Assistance - testify',
	'Assistance - risk',
	'Young offender',
	'Medical conditions',
	'Family illness',
	'Rehabilitation programme',
}

PLEA_STAGE_MODEL_MAP = {
	('High Court', 'Up to committal'): 'Plead guilty (earliest opportunity)',
	('High Court', 'After committal'): 'Plead guilty (before trial dates are set)',
	('High Court', 'After dates fixed'): 'Plead guilty (before trial starts)',
	('High Court', 'First day'): 'Plead guilty (first day of trial)',
	('High Court', 'During trial'): 'Plead guilty (during the trial)',
	('District Court', 'Plea day'): 'Plead guilty (earliest opportunity)',
	('District Court', 'After dates fixed'): 'Plead guilty (before trial starts)',
	('District Court', 'First day'): 'Plead guilty (first day of trial)',
	('District Court', 'During trial'): 'Plead guilty (during the trial)',
}

def trial_model_drugs(drugs_json):
	model_drugs = []
	unsupported_drugs = []
	for drug in json.loads(drugs_json):
		drug_type = drug.get('drug_type')
		if not drug_type:
			continue
		quantity, invalid = clean_quantity(drug.get('quantity'))
		if invalid or quantity <= 0:
			continue
		if drug_type == 'Other':
			other = (drug.get('other_drug_type') or '').strip().lower()
			if 'midazolam' in other:
				model_drugs.append({'type': 'Midazolam', 'quantity': quantity, 'variant': 'powder'})
			else:
				unsupported_drugs.append(f'{drug_type}({other})')
			continue
		model_type = DRUG_VERIFIED_TO_MODEL.get(drug_type)
		if model_type is None:
			unsupported_drugs.append(drug_type)
		else:
			model_drugs.append({'type': model_type, 'quantity': quantity})
	return model_drugs, unsupported_drugs

def build_model_input(row):
	drugs, unsupported_drugs = trial_model_drugs(row['drugs_json'])
	model_aggravating = [
		AGGRAVATING_MODEL_MAP[factor]
		for factor in row['canonical_aggravating_factors']
		if factor in AGGRAVATING_MODEL_MAP
	]
	unsupported_aggravating = [
		factor
		for factor in row['canonical_aggravating_factors']
		if factor not in AGGRAVATING_MODEL_MAP
		and factor not in {'Role of the defendant', 'Cross-border trafficking'}
	]
	model_mitigating = [
		factor
		for factor in row['canonical_mitigating_factors']
		if factor in MITIGATING_MODEL_FACTORS
	]
	unsupported_mitigating = [
		factor
		for factor in row['canonical_mitigating_factors']
		if factor not in MITIGATING_MODEL_FACTORS
	]
	plea = json.loads(row['guilty_plea_json'])
	guilty_plea = None
	plea_status = 'not guilty'
	if plea.get('pleaded_guilty'):
		stage = plea.get('high_court_stage') or plea.get('district_court_stage') or 'Unknown'
		court = plea.get('court_type')
		guilty_plea = PLEA_STAGE_MODEL_MAP.get((court, stage))
		plea_status = 'mapped' if guilty_plea is not None else 'unmapped stage'
	return {
		'drugs': drugs,
		'unsupported_drugs': unsupported_drugs,
		'defendant_role': (
			row['model_role'] if isinstance(row['model_role'], str) else None
		),
		'additional_circumstances': (
			row['model_circumstances']
			if isinstance(row['model_circumstances'], list)
			else []
		),
		'guilty_plea': guilty_plea,
		'plea_status': plea_status,
		'aggravating_factors': model_aggravating,
		'unsupported_aggravating': unsupported_aggravating,
		'mitigating_factors': model_mitigating,
		'unsupported_mitigating': unsupported_mitigating,
	}


In [5]:
rows = []
for _index, row in trial_rows.iterrows():
	model_input = build_model_input(row)
	if model_input['drugs']:
		result = predict_sentence(
			model_input['drugs'],
			model_input['defendant_role'],
			model_input['additional_circumstances'],
			model_input['guilty_plea'],
			model_input['aggravating_factors'],
			model_input['mitigating_factors'],
		)
	else:
		result = None
	actual = row['final_sentence_months']
	covered = result is not None and not model_input['unsupported_drugs']
	predicted = result['final_sentence_months'] if covered else None
	rows.append({
		'case_id': row['case_id'],
		'neutral_citation': row['neutral_citation'],
		'trial_index': row['trial_index'],
		'charge_no': row['charge_no'],
		'defendant_id': row['defendant_id'],
		'drugs_json': row['drugs_json'],
		'primary_role': model_input['defendant_role'],
		'additional_circumstances': ' | '.join(model_input['additional_circumstances']),
		'aggravating_factors': ' | '.join(model_input['aggravating_factors']),
		'unsupported_aggravating': ' | '.join(model_input['unsupported_aggravating']),
		'mitigating_factors': ' | '.join(model_input['mitigating_factors']),
		'unsupported_mitigating': ' | '.join(model_input['unsupported_mitigating']),
		'plea_status': model_input['plea_status'],
		'guilty_plea': model_input['guilty_plea'],
		'unsupported_drugs': ' | '.join(model_input['unsupported_drugs']),
		'predicted_starting_point_months': result['starting_point_months'] if covered else None,
		'predicted_role_months': result['role_months'] if covered else None,
		'predicted_aggravating_months': result['aggravating_months'] if covered else None,
		'predicted_mitigating_months': result['mitigating_months'] if covered else None,
		'predicted_plea_months': result['plea_months'] if covered else None,
		'predicted_final_months': predicted,
		'actual_final_months': actual,
		'covered': covered,
	})

predictions = pd.DataFrame(rows)
covered_df = predictions.loc[predictions['covered']].copy()
covered_df['difference_months'] = (
	covered_df['predicted_final_months'] - covered_df['actual_final_months']
)
covered_df['absolute_difference_months'] = covered_df['difference_months'].abs()

print(f"eligible trials: {len(predictions)}")
print(f"covered by the model: {len(covered_df)} ({len(covered_df) / len(predictions):.1%})")
print(f"unsupported drugs: {predictions['unsupported_drugs'].astype(bool).sum()}")


eligible trials: 2599
covered by the model: 2549 (98.1%)
unsupported drugs: 42


In [6]:
def summarize(group, label):
	if len(group) == 0:
		return {'group': label, 'trials': 0}
	error = group['difference_months']
	absolute = group['absolute_difference_months']
	actual = group['actual_final_months']
	positive_actual = actual[actual > 0]
	positive_error = error.loc[positive_actual.index]
	positive_absolute = absolute.loc[positive_actual.index]
	within_25 = positive_absolute <= positive_actual * 0.25
	case_accuracy = np.maximum(1 - positive_absolute / positive_actual, 0)
	return {
		'group': label,
		'trials': len(group),
		'mae_months': round(absolute.mean(), 2),
		'median_abs_error_months': round(absolute.median(), 2),
		'rmse_months': round(np.sqrt((error**2).mean()), 2),
		'mean_signed_error_months': round(error.mean(), 2),
		'exact_match_rate': round((error == 0).mean() * 100, 1),
		'within_12m_rate': round((absolute <= 12).mean() * 100, 1),
		'within_24m_rate': round((absolute <= 24).mean() * 100, 1),
		'within_25pct_rate': round(within_25.mean() * 100, 1),
		'mape_pct': round((positive_absolute / positive_actual).mean() * 100, 1),
		'mean_case_accuracy_pct': round(case_accuracy.mean() * 100, 1),
	}

drug_count = covered_df['drugs_json'].map(
	lambda value: 'multiple' if len(json.loads(value)) > 1 else 'single'
)

groups = [
	('overall', covered_df['difference_months'].notna()),
	('single drug', drug_count.eq('single')),
	('multiple drugs', drug_count.eq('multiple')),
	('no role', covered_df['primary_role'].isna()),
	('role from workbook', covered_df['primary_role'].notna()),
	('not guilty', covered_df['plea_status'].eq('not guilty')),
	('guilty - plea mapped', covered_df['plea_status'].eq('mapped')),
	('guilty - stage unmapped', covered_df['plea_status'].eq('unmapped stage')),
	('any aggravating applied', covered_df['aggravating_factors'].astype(bool)),
	('any mitigating applied', covered_df['mitigating_factors'].astype(bool)),
	('assistance applied', covered_df['mitigating_factors'].str.contains('Assistance - ')),
	('unsupported input present', (
		covered_df['unsupported_drugs'].astype(bool)
		| covered_df['unsupported_aggravating'].astype(bool)
		| covered_df['unsupported_mitigating'].astype(bool)
		| covered_df['plea_status'].eq('unmapped stage')
	)),
	('all inputs supported', (
		~covered_df['unsupported_drugs'].astype(bool)
		& ~covered_df['unsupported_aggravating'].astype(bool)
		& ~covered_df['unsupported_mitigating'].astype(bool)
		& ~covered_df['plea_status'].eq('unmapped stage')
	)),
]

metrics = pd.DataFrame([
	summarize(covered_df[mask], label)
	for label, mask in groups
 if mask.sum() > 0
])

display(metrics.set_index('group'))


,trials,mae_months,median_abs_error_months,rmse_months,mean_signed_error_months,exact_match_rate,within_12m_rate,within_24m_rate,within_25pct_rate,mape_pct,mean_case_accuracy_pct
group,,,,,,,,,,,
overall,2549,11.11,3.94,18.91,1.01,0.1,67.3,84.0,75.4,19.8,82.2
single drug,1643,11.69,3.89,19.93,1.17,0.1,65.8,82.6,76.9,20.2,82.4
multiple drugs,906,10.05,4.11,16.90,0.71,0.2,70.0,86.6,72.7,18.9,81.8
no role,2445,11.04,3.87,18.90,0.95,0.1,67.7,84.2,75.5,19.8,82.3
role from workbook,104,12.90,8.44,19.08,2.35,0.0,57.7,79.8,73.1,19.0,81.0
not guilty,197,18.13,5.59,31.66,-12.08,0.5,62.9,71.1,91.3,14.1,88.6
guilty - plea mapped,1852,7.24,2.10,14.03,-3.18,0.1,82.0,89.8,92.9,9.9,90.6
guilty - stage unmapped,500,22.68,19.48,26.44,21.68,0.0,14.6,67.6,4.2,58.7,48.7
any aggravating applied,913,10.09,3.74,17.04,1.89,0.1,69.8,86.0,74.2,17.7,82.4


In [7]:
report_path = notebook_dir / "full_model_evaluation_analysis.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
	metrics.set_index("group").to_excel(writer, sheet_name="metrics", index=True)
	covered_df.drop(columns=["drugs_json"]).round(2).to_excel(
		writer, sheet_name="predictions", index=False
	)
print("Wrote", report_path)


Wrote /Users/cxiang/Projects/drug-trafficing-sentence-predictor/notebooks/predictionModel/full_model_evaluation_analysis.xlsx


## Reading the results

- `metrics` — prediction error for the **full** model (guideline starting point,
  role / aggravating / mitigating adjustments, plea discount last) against verified
  final sentences. MAE / median absolute error in months, plus exact-match and
  within-12-month rates.
- `predictions` — one row per eligible trial with the model inputs, stage-by-stage
  predicted months, the predicted vs actual final sentence and the difference.
- Trials with an unsupported drug (no guideline curve) are excluded from coverage;
  factors and plea stages outside the model vocabulary are recorded but not applied.
- Roles come from `Role Sentence Adjustments_updated 2026.07.30.xlsx` (the verified
  trials carry no sentencing role profile).
